# Professional Data Cleaning Notebook
## Problem Detection → Interpretation → Solution → Code

This notebook serves as a practical interview cheat sheet and project template for Data Analysts, Data Scientists, and ML Engineers.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("your_dataset.csv")
df.head(10)
df.tail(10)


## 1. Missing Values Handling

### Problem Detection

```python
df.isnull().sum()
df.isnull().mean()*100
```

### Interpretation

- Few missing values (<5%) → usually safe to drop.
- Numeric columns → Mean/Median.
- Categorical columns → Mode.
- Time series → Forward Fill / Backward Fill.

### Solutions

- Drop rows
- Drop columns
- Mean Imputation
- Median Imputation
- Mode Imputation
- Forward Fill
- Backward Fill


In [ ]:
# Drop rows
df.dropna(inplace=True)

# Mean
df["salary"] = df["salary"].fillna(df["salary"].mean())

# Median
df["age"] = df["age"].fillna(df["age"].median())

# Mode
df["city"] = df["city"].fillna(df["city"].mode()[0])

# Forward Fill
df["sales"] = df["sales"].ffill()

# Backward Fill
df["sales"] = df["sales"].bfill()


## 2. Duplicate Data Removal

### Problem Detection

```python
df.duplicated().sum()
```

### Interpretation

- Duplicate records create bias.
- Same customer appearing multiple times can distort analysis.

### Solutions

- Remove exact duplicates.
- Remove duplicate IDs.
- Keep latest record only.


In [ ]:
# Exact duplicates
df.drop_duplicates(inplace=True)

# Duplicate IDs
df.drop_duplicates(
    subset=["customer_id"],
    keep="first"
)


## 3. Wrong Format Fix

### Problem Detection

```python
df.dtypes
```

Common Issues:

- Dates stored as object
- Numbers stored as strings
- Currency/unit symbols inside values

### Solutions

- Convert date formats
- Convert strings to numeric
- Normalize categorical values


In [ ]:
# Date conversion
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

# Currency cleanup
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace("$","",regex=False)
)

df["price"] = pd.to_numeric(
    df["price"],
    errors="coerce"
)

# Category normalization
df["country"] = df["country"].replace({
    "United States":"USA",
    "U.S.A":"USA"
})


## 4. Inconsistent Labels

### Problem Detection

```python
df["city"].unique()
df["brand"].unique()
```

### Interpretation

Same entity represented differently.

Example:

- Apple
- apple
- APPLE

### Solutions

- Case normalization
- Spelling correction
- Synonym mapping


In [ ]:
# Lowercase
df["brand"] = df["brand"].str.lower()

# Spelling correction
df["city"] = df["city"].replace({
    "Dhka":"Dhaka"
})

# Synonyms
df["vehicle"] = df["vehicle"].replace({
    "automobile":"car"
})


## 5. Outlier Detection & Handling

### Problem Detection

Methods:

- Boxplot
- IQR
- Z-Score
- Domain Rules

Example:

Age < 0
Salary > 10M

### Solutions

- Remove
- Cap (Winsorization)
- Log Transform


In [ ]:
Q1 = df["salary"].quantile(0.25)
Q3 = df["salary"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

# Detect
outliers = df[
    (df["salary"] < lower) |
    (df["salary"] > upper)
]

# Remove
df = df[
    (df["salary"] >= lower) &
    (df["salary"] <= upper)
]

# Cap
df["salary"] = df["salary"].clip(
    lower,
    upper
)

# Log Transform
df["salary"] = np.log1p(df["salary"])


## 6. Noisy Data Filtering

### Problem Detection

Examples:

- Invalid ages
- Typing mistakes
- Random sensor noise
- Blurry images
- Irrelevant text samples

### Solutions

- Rule-based filtering
- Threshold filtering
- Quality filtering


In [ ]:
# Invalid age
df = df[df["age"] >= 0]

# Remove short reviews
df = df[df["review"].str.len() > 5]

# Image example
# if blur_score < 100:
#     discard_image()


## 7. Data Type Consistency Check

### Problem Detection

```python
df.dtypes
```

Common Problems:

- int stored as object
- boolean stored as string
- category stored as object

### Solutions

Convert to correct data types.


In [ ]:
df["age"] = df["age"].astype(int)

df["active"] = df["active"].map({
    "True":True,
    "False":False
})

df["city"] = df["city"].astype("category")


## 8. Schema Validation

### Problem Detection

Check:

- Missing columns
- Extra columns
- Wrong column names

### Solutions

Validate schema before pipeline execution.


In [ ]:
required_cols = [
    "id",
    "name",
    "age"
]

missing = (
    set(required_cols)
    - set(df.columns)
)

extra = (
    set(df.columns)
    - set(required_cols)
)

print("Missing:", missing)
print("Extra:", extra)


## 9. Data Range Validation

### Problem Detection

Examples:

- age > 150
- price < 0
- probability > 1

### Solutions

Apply business/domain rules.


In [ ]:
# Age
df = df[
    (df["age"] >= 0) &
    (df["age"] <= 150)
]

# Price
df = df[df["price"] >= 0]

# Probability
df = df[
    (df["probability"] >= 0) &
    (df["probability"] <= 1)
]


## 10. Imbalanced Data Check

### Problem Detection

```python
df["target"].value_counts()
df["target"].value_counts(normalize=True)*100
```

Example:

Class 0 → 95%
Class 1 → 5%

Interpretation:

Highly Imbalanced Dataset

### Solutions

1. Oversampling
2. Undersampling
3. SMOTE
4. Class Weighting
5. Better Evaluation Metrics


In [ ]:
# Check distribution
print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True)*100)

# ---------------------
# Oversampling
# ---------------------
from sklearn.utils import resample

majority = df[df.target == 0]
minority = df[df.target == 1]

minority_up = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

df_balanced = pd.concat(
    [majority, minority_up]
)

# ---------------------
# Undersampling
# ---------------------

majority_down = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

df_under = pd.concat(
    [majority_down, minority]
)

# ---------------------
# SMOTE
# ---------------------

from imblearn.over_sampling import SMOTE

X = df.drop("target", axis=1)
y = df["target"]

smote = SMOTE(random_state=42)

X_resampled, y_resampled = (
    smote.fit_resample(X, y)
)

# ---------------------
# Class Weight
# ---------------------

# LogisticRegression(
#     class_weight="balanced"
# )

# ---------------------
# Better Metrics
# ---------------------

# precision_score
# recall_score
# f1_score
# roc_auc_score
